# Differential Gene Expression Analysis
## Breast Cancer Tumor vs. Adjacent Normal Tissue

**Author:** Szonja Lippert — MSc Bioinformatics, Wageningen University  
**Dataset:** Modelled on GEO:GSE7904 (Affymetrix HG-U133Plus2, breast cancer)  
**Goal:** Identify genes significantly upregulated or downregulated in tumor tissue compared to normal tissue, using proper statistical testing and multiple-testing correction.

---

### Why this matters
Differential gene expression analysis is the cornerstone of transcriptomics. By comparing expression profiles between disease and healthy tissue, we can:
- Identify **biomarkers** for diagnosis or prognosis
- Discover **therapeutic targets** (e.g., overexpressed oncogenes)
- Understand the **molecular mechanisms** driving disease

## 1. Setup & Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

# Display settings
pd.set_option('display.max_columns', 10)
pd.set_option('display.float_format', '{:.4f}'.format)
plt.rcParams['figure.dpi'] = 120

print('Libraries loaded ✓')

## 2. Load Expression Data

The expression matrix is in **log₂ scale** — this is standard for Affymetrix microarray data after RMA normalisation.  
Each row is a gene, each column is a sample.

In [ ]:
# Load the expression matrix
expr = pd.read_csv('data/expression_matrix.csv', index_col=0)
groups = pd.read_csv('data/sample_groups.csv')

tumor_samples  = groups[groups['group'] == 'Tumor']['sample'].tolist()
normal_samples = groups[groups['group'] == 'Normal']['sample'].tolist()

print(f'Expression matrix shape: {expr.shape[0]} genes × {expr.shape[1]} samples')
print(f'Tumor samples:  {len(tumor_samples)}')
print(f'Normal samples: {len(normal_samples)}')
print()
expr.head()

In [ ]:
# Quick sanity check: housekeeping genes should be similar between groups
for gene in ['GAPDH', 'ACTB', 'RPLP0']:
    t_mean = expr.loc[gene, tumor_samples].mean()
    n_mean = expr.loc[gene, normal_samples].mean()
    print(f'{gene}: Tumor={t_mean:.2f}, Normal={n_mean:.2f}, diff={t_mean-n_mean:+.3f} ← should be ~0')

## 3. Differential Expression Testing

### Statistical approach

For each gene, we test whether the **mean expression differs** between tumor and normal.

We use **Welch's t-test** (not Student's t-test) because:
1. We have unequal sample sizes (12 vs. 10)
2. We cannot assume equal variance between groups
3. Welch's is more robust and is now the default in most DGE tools

The **log₂ fold change** is simply the difference in means (since data is already log₂ scaled):  
$$\text{log}_2\text{FC} = \bar{x}_{\text{tumor}} - \bar{x}_{\text{normal}}$$

In [ ]:
results = []

for gene in expr.index:
    tumor_vals  = expr.loc[gene, tumor_samples].values.astype(float)
    normal_vals = expr.loc[gene, normal_samples].values.astype(float)
    
    log2fc = np.mean(tumor_vals) - np.mean(normal_vals)
    t_stat, p_val = stats.ttest_ind(tumor_vals, normal_vals, equal_var=False)
    
    results.append({
        'gene': gene,
        'log2FC': round(log2fc, 4),
        'mean_tumor': round(np.mean(tumor_vals), 4),
        'mean_normal': round(np.mean(normal_vals), 4),
        'p_value': p_val,
        't_statistic': round(t_stat, 4)
    })

results_df = pd.DataFrame(results)
print(f'Tested {len(results_df)} genes')
results_df.head(10)

## 4. Multiple Testing Correction (BH FDR)

Testing 358 genes simultaneously means we expect ~18 false positives at α=0.05 by chance alone.  

The **Benjamini-Hochberg (BH) procedure** controls the False Discovery Rate (FDR):  
it adjusts each p-value based on its rank among all tests, so that the *expected proportion of false discoveries* among significant results is controlled at 5%.

$$p_{\text{adj},i} = p_i \times \frac{n}{\text{rank}_i}$$

In [ ]:
# Sort by p-value ascending
results_df = results_df.sort_values('p_value').reset_index(drop=True)
n = len(results_df)

results_df['rank'] = range(1, n + 1)
results_df['adj_p_value'] = (results_df['p_value'] * n / results_df['rank']).clip(upper=1.0)

# Enforce monotonicity (required by BH)
adj_p = results_df['adj_p_value'].values.copy()
for i in range(len(adj_p) - 2, -1, -1):
    adj_p[i] = min(adj_p[i], adj_p[i + 1])
results_df['adj_p_value'] = adj_p

# Significance flags
results_df['neg_log10_p'] = -np.log10(results_df['adj_p_value'].clip(lower=1e-15))
results_df['significant']  = (results_df['adj_p_value'] < 0.05) & (results_df['log2FC'].abs() >= 1.0)
results_df['direction'] = np.where(
    ~results_df['significant'], 'Not Significant',
    np.where(results_df['log2FC'] > 0, 'Upregulated in Tumor', 'Downregulated in Tumor')
)

print('Significant DEGs (adj.p<0.05, |log2FC|>=1):', results_df['significant'].sum())
print('  Upregulated:  ', (results_df['direction']=='Upregulated in Tumor').sum())
print('  Downregulated:', (results_df['direction']=='Downregulated in Tumor').sum())

## 5. Volcano Plot

The volcano plot is the standard visualisation for DGE results.  
- **X-axis**: log₂ fold change — effect size  
- **Y-axis**: −log₁₀(adjusted p-value) — statistical confidence  
- **Top-right**: significantly upregulated in tumor  
- **Top-left**: significantly downregulated in tumor

In [ ]:
fig, ax = plt.subplots(figsize=(9, 7))

color_map = {
    'Upregulated in Tumor':   '#E8334A',
    'Downregulated in Tumor': '#2E86AB',
    'Not Significant':        '#CCCCCC',
}

real = results_df[~results_df['gene'].str.startswith('LOC')]
for direction, grp in real.groupby('direction'):
    size = 55 if direction != 'Not Significant' else 18
    alpha = 0.85 if direction != 'Not Significant' else 0.35
    ax.scatter(grp['log2FC'], grp['neg_log10_p'], c=color_map[direction],
               s=size, alpha=alpha, linewidths=0, label=direction)

# Threshold lines
ax.axhline(-np.log10(0.05), color='grey', ls='--', lw=1)
ax.axvline(-1.0, color='grey', ls='--', lw=1)
ax.axvline(+1.0, color='grey', ls='--', lw=1)

# Label top genes
sig = real[real['significant']]
for _, row in pd.concat([
    sig[sig['direction']=='Upregulated in Tumor'].nlargest(8, 'log2FC'),
    sig[sig['direction']=='Downregulated in Tumor'].nsmallest(8, 'log2FC')
]).iterrows():
    ax.annotate(row['gene'], xy=(row['log2FC'], row['neg_log10_p']),
                xytext=(row['log2FC'] + (0.3 if row['log2FC']>0 else -0.3), row['neg_log10_p']+0.3),
                fontsize=7.5, fontweight='bold',
                arrowprops=dict(arrowstyle='-', color='grey', lw=0.6))

ax.set_xlabel('Log₂ Fold Change (Tumor / Normal)', fontsize=12)
ax.set_ylabel('−log₁₀ (Adjusted p-value)', fontsize=12)
ax.set_title('Volcano Plot: Breast Cancer DGE', fontsize=14, fontweight='bold')
ax.legend(loc='lower right', framealpha=0.9)
ax.spines[['top', 'right']].set_visible(False)
plt.tight_layout()
plt.show()

## 6. Top DEGs Table

In [ ]:
sig_genes = results_df[results_df['significant'] & ~results_df['gene'].str.startswith('LOC')]

print('TOP 10 UPREGULATED IN TUMOR')
display(sig_genes[sig_genes['direction']=='Upregulated in Tumor']
        .nlargest(10, 'log2FC')[['gene','log2FC','adj_p_value','mean_tumor','mean_normal']])

print('\nTOP 10 DOWNREGULATED IN TUMOR')
display(sig_genes[sig_genes['direction']=='Downregulated in Tumor']
        .nsmallest(10, 'log2FC')[['gene','log2FC','adj_p_value','mean_tumor','mean_normal']])

## 7. PCA — Unsupervised Validation

PCA on the top 200 most variable genes, computed via SVD.  
If the DGE results are real, tumor and normal should separate on PC1 — *without being told which group each sample belongs to*.

In [ ]:
from numpy.linalg import svd

real_genes = [g for g in expr.index if not g.startswith('LOC')]
top_var = expr.loc[real_genes].var(axis=1).nlargest(200).index
X = expr.loc[top_var].T.values
X_c = X - X.mean(axis=0)

U, S, Vt = svd(X_c, full_matrices=False)
PC1, PC2 = U[:,0]*S[0], U[:,1]*S[1]
explained = (S**2)/(S**2).sum()

fig, ax = plt.subplots(figsize=(7, 6))
samples = expr.columns.tolist()
colors  = ['#E8334A' if s in tumor_samples else '#2E86AB' for s in samples]
ax.scatter(PC1, PC2, c=colors, s=80, edgecolors='white', linewidths=0.8, alpha=0.9, zorder=3)

ax.set_xlabel(f'PC1 ({explained[0]*100:.1f}% variance)', fontsize=11)
ax.set_ylabel(f'PC2 ({explained[1]*100:.1f}% variance)', fontsize=11)
ax.set_title('PCA — Top 200 Variable Genes', fontsize=13, fontweight='bold')

legend_handles = [
    mpatches.Patch(color='#E8334A', label=f'Tumor (n={len(tumor_samples)})'),
    mpatches.Patch(color='#2E86AB', label=f'Normal (n={len(normal_samples)})')
]
ax.legend(handles=legend_handles)
ax.spines[['top','right']].set_visible(False)
plt.tight_layout()
plt.show()

print(f'PC1 explains {explained[0]*100:.1f}% of variance — tumor/normal separation is the dominant signal')

## 8. Biological Interpretation

The top upregulated genes fall into well-known cancer hallmark pathways:

| Pathway | Key Genes | Clinical Relevance |
|---------|-----------|-------------------|
| **Proliferation** | MKI67, TOP2A, CCNB1, CDK1, CDC20 | Ki-67 (MKI67) is a clinical proliferation index |
| **EMT / Invasion** | TWIST1, SNAI1, VIM, ZEB1, MMP9 | Associated with metastatic potential |
| **HER2 signalling** | ERBB2 | Clinically targetable oncogene (trastuzumab) |
| **Hypoxia** | HIF1A, VEGFA, LDHA | Warburg effect; angiogenesis driver |
| **Cell adhesion lost** | CDH1↓, EPCAM↓, CLDN4↓ | Loss of epithelial identity in EMT |
| **Apoptosis suppressed** | BAX↓, CASP3↓, FAS↓ | Evasion of programmed cell death |

The downregulation of **ESR1** and **PGR** suggests an ER-negative phenotype — clinically important as these tumors require chemotherapy rather than hormone therapy.

## 9. Save Results

In [ ]:
results_df.to_csv('results/dge_results_notebook.csv', index=False)
sig_genes.to_csv('results/dge_significant_notebook.csv', index=False)

print('Results saved:')
print(f'  results/dge_results_notebook.csv  — all {len(results_df)} genes')
print(f'  results/dge_significant_notebook.csv — {len(sig_genes)} significant DEGs')

---
## Summary

This notebook demonstrates a complete DGE workflow:

1. **Data loading** — log₂ microarray expression matrix
2. **Statistical testing** — Welch's t-test for each gene
3. **Multiple testing correction** — Benjamini-Hochberg FDR
4. **Visualisation** — Volcano plot, PCA
5. **Biological interpretation** — pathway context for top hits

The results are consistent with known breast cancer biology, validating both the methodology and the biological signal in the data.

**Next steps:** GSEA/ORA pathway enrichment · survival correlation (TCGA-BRCA) · DESeq2 on RNA-seq data

---
*Szonja Lippert · MSc Bioinformatics, WUR · 2025*